### RaschPy conditional Rasch analysis: worked example (PCM)

A conditional analysis in the style of Christensen, Kreiner & Mesbah (2013): every step conditions on each person's total score, so the person parameters drop out. The steps are CML estimation, overall fit, item fit, local dependence and dimensionality, DIF, person fit and targeting, then a choice between the partial credit and rating scale models.

#### Data

This example uses the DESC-II depression screening data from **iarm**, an R package for Rasch analysis (Mueller, 2022; https://CRAN.R-project.org/package=iarm). The data are 799 patients' responses to 10 items scored from 0 (never) to 4 (always), with each patient's indication group, gender and age group. To run the notebook, export the data once from R into the working directory:

```r
install.packages("iarm")
write.csv(iarm::desc2, "desc2.csv", row.names = FALSE)
```

In [1]:
import os
import numpy as np
import pandas as pd
from raschpy import PCM, RSM

os.chdir('my_working_directory')

desc2 = pd.read_csv('desc2.csv')
items = [c for c in desc2.columns if c.startswith('DESC')]
responses = desc2[items]
exogenous = desc2[['group', 'gender', 'agegroup']]

#### 1. CML estimation

`estimation='cml'` gives conditional maximum likelihood estimates. For the PCM, CML has a ridge by default; `ridge=0` gives plain CML. Its estimates, infit and outfit mean squares, item-restscore gammas and standard errors, and Martin-Löf results agree with iarm's to the decimals it prints. Item-restscore *p*-values differ, because RaschPy tests against a simulated distribution rather than iarm's normal approximation; section 3 also computes iarm's version.

In [2]:
pcm = PCM(responses,
          max_score_vector=[4] * len(items),
          exogenous=exogenous)
pcm.calibrate(estimation='cml',
              ridge=0)
pcm.std_errors()  # sandwich standard errors

round(pd.DataFrame({'Location': pcm.items, 'SE': pcm.item_se}), 3)

,Location,SE
DESC_2_1,0.117,0.060
DESC_2_2,0.452,0.070
DESC_2_3,-0.891,0.054
DESC_2_4,-0.564,0.055
DESC_2_5,0.347,0.056
DESC_2_6,0.148,0.062
DESC_2_7,-0.057,0.053
DESC_2_8,-0.220,0.053
DESC_2_9,-0.552,0.057
DESC_2_10,1.220,0.110


In [3]:
round(pcm.thresholds_uncentred, 3)

,1,2,3,4
DESC_2_1,-0.945,-0.779,0.667,1.524
DESC_2_2,-0.589,-0.540,0.980,1.959
DESC_2_3,-3.414,-1.647,0.096,1.399
DESC_2_4,-2.618,-1.069,0.072,1.359
DESC_2_5,-0.311,-0.391,0.393,1.697
DESC_2_6,-1.610,-0.429,0.482,2.149
DESC_2_7,-1.177,-0.824,0.424,1.351
DESC_2_8,-2.121,-1.006,0.369,1.876
DESC_2_9,-2.390,-1.438,-0.084,1.704
DESC_2_10,0.769,0.385,1.670,2.057


Standard errors from the inverse Hessian instead, as standard for most CML software:

In [4]:
pcm.std_errors(se_method='hessian')
hessian_se = pcm.item_se
pcm.std_errors()

round(pd.DataFrame({'Sandwich': pcm.item_se, 'Hessian': hessian_se}), 3)

,Sandwich,Hessian
DESC_2_1,0.060,0.058
DESC_2_2,0.070,0.066
DESC_2_3,0.054,0.058
DESC_2_4,0.055,0.056
DESC_2_5,0.056,0.060
DESC_2_6,0.062,0.065
DESC_2_7,0.053,0.056
DESC_2_8,0.053,0.060
DESC_2_9,0.057,0.057
DESC_2_10,0.110,0.086


#### 2. Overall fit

Andersen's conditional likelihood ratio test, splitting the persons by total score and by gender. In the score split, every item has a category that no one in the low-score group used, so plain CML has no estimates for that group. `unused_categories='null'` leaves those categories' thresholds out of that group's fit, as iarm does. The chi-square *p*-value is then liberal, so `p_method='monte-carlo'` gives a calibrated one.

In [5]:
pcm.global_fit(method='andersen',
               split='score',
               unused_categories='null',
               p_method='monte-carlo',
               seed=1)
pcm.global_fit_summary

Andersen LR Monte Carlo p-value: 1/500 samples | elapsed 0s | about 59s left

Andersen LR Monte Carlo p-value: 10/500 samples | elapsed 1s | about 31s left

Andersen LR Monte Carlo p-value: 18/500 samples | elapsed 1s | about 32s left

Andersen LR Monte Carlo p-value: 28/500 samples | elapsed 2s | about 29s left

Andersen LR Monte Carlo p-value: 38/500 samples | elapsed 2s | about 28s left

Andersen LR Monte Carlo p-value: 47/500 samples | elapsed 3s | about 27s left

... (50 outputs left out)

Andersen LR Monte Carlo p-value: 498/500 samples | elapsed 30s | about 0s left

Andersen LR Monte Carlo p-value: 500/500 samples | elapsed 30s | about 0s left

Andersen LR Monte Carlo p-value: 500/500 samples | elapsed 30s | about 0s left | done

method                                                                  andersen
split                                                                      score
statistic                                                              38.682936
df                                                                            21
split_prop                                                                   0.5
log-likelihoods                {'low': -1117.5775857761469, 'high': -3715.953...
p_method                                                             monte-carlo
p-value                                                                 0.063872
p-value (chi-square)                                                    0.010707
Monte Carlo samples                                                          500
Monte Carlo samples skipped                                                    0
Monte Carlo stopped early                                                  False
reject                      

In [6]:
pcm.global_fit(method='andersen',
               split='exogenous',
               covariate='gender',
               seed=1)
pcm.global_fit_summary

method                                                                  andersen
split                                                                  exogenous
statistic                                                              79.436582
df                                                                            39
covariate                                                                 gender
log-likelihoods                {'female': -2435.961521532983, 'male': -2371.8...
p_method                                                                    chi2
p-value                                                                  0.00014
p-value (chi-square)                                                     0.00014
Monte Carlo samples                                                         None
Monte Carlo samples skipped                                                 None
Monte Carlo stopped early                                                   None
reject                      

Martin-Löf's test of unidimensionality, splitting the items at their median location. Most cells of its score table are sparse here, so the Monte Carlo *p*-value is used:

In [7]:
pcm.global_fit(method='martin-lof',
               p_method='monte-carlo',
               seed=1)
pcm.global_fit_summary

Martin-Lof Monte Carlo p-value: 1/500 samples | elapsed 0s | about 22s left

Martin-Lof Monte Carlo p-value: 12/500 samples | elapsed 1s | about 23s left

Martin-Lof Monte Carlo p-value: 23/500 samples | elapsed 1s | about 23s left

Martin-Lof Monte Carlo p-value: 34/500 samples | elapsed 2s | about 22s left

Martin-Lof Monte Carlo p-value: 45/500 samples | elapsed 2s | about 21s left

Martin-Lof Monte Carlo p-value: 56/500 samples | elapsed 3s | about 21s left

... (38 outputs left out)

Martin-Lof Monte Carlo p-value: 494/500 samples | elapsed 23s | about 0s left

Martin-Lof Monte Carlo p-value: 500/500 samples | elapsed 23s | about 0s left

Martin-Lof Monte Carlo p-value: 500/500 samples | elapsed 23s | about 0s left | done

method                                                                 martin-lof
split                                                                      median
statistic                                                                351.3902
df                                                                            399
subsets                         {'A': ['DESC_2_3', 'DESC_2_4', 'DESC_2_7', 'DE...
log-likelihoods                 {'A': -2104.515831874477, 'B': -1710.990510840...
p_method                                                              monte-carlo
p-value                                                                  0.001996
p-value (chi-square)                                                      0.95851
Monte Carlo samples                                                           500
Monte Carlo samples skipped                                                     0
Monte Carlo stopped early                                                   False
reject          

#### 3. Item fit

Conditional infit and outfit, item-restscore and the conditional item chi-square Q1, with *p*-values and cutoffs from 200 datasets simulated from the model given each person's total, and Westfall–Young familywise *p*-values:

In [8]:
pcm.fit_statistics(conditional=True,
                   bootstrap_samples=200,
                   seed=1)
pcm.item_stats_df(full=True)

,Estimate,SE,2.5%,97.5%,Count,Facility,Infit MS,Infit Z,Outfit MS,Outfit Z,PM corr,Exp PM corr,Observed gamma,Expected gamma,Restscore p (BH),Q1,Q1 df,Q1 p (BH),Infit p (Westfall-Young),Outfit p (Westfall-Young)
DESC_2_1,0.117,0.060,-0.002,0.235,799,0.230,1.097,1.540,1.235,2.349,0.771,0.737,0.776,0.797,0.187,6.108,9,0.829,0.388,0.090
DESC_2_2,0.452,0.070,0.315,0.590,799,0.195,1.101,1.538,1.155,1.361,0.752,0.718,0.790,0.798,0.627,10.168,9,0.752,0.388,0.687
DESC_2_3,-0.891,0.054,-0.998,-0.785,799,0.368,0.892,-2.107,0.908,-1.868,0.825,0.725,0.801,0.780,0.187,8.802,9,0.752,0.229,0.353
DESC_2_4,-0.564,0.055,-0.673,-0.455,799,0.322,1.065,1.180,1.068,1.272,0.802,0.743,0.773,0.782,0.453,5.941,9,0.829,0.478,0.687
DESC_2_5,0.347,0.056,0.237,0.456,799,0.197,0.898,-1.528,0.914,-0.641,0.778,0.720,0.827,0.808,0.276,8.077,9,0.752,0.468,0.801
DESC_2_6,0.148,0.062,0.027,0.269,799,0.241,0.979,-0.348,1.013,0.207,0.788,0.731,0.779,0.783,0.715,4.323,9,0.889,0.662,0.930
DESC_2_7,-0.057,0.053,-0.160,0.047,799,0.250,0.910,-1.508,0.849,-1.845,0.815,0.744,0.823,0.795,0.138,14.055,9,0.444,0.478,0.353
DESC_2_8,-0.220,0.053,-0.325,-0.116,799,0.287,0.799,-3.884,0.797,-3.699,0.843,0.740,0.834,0.782,0.000,13.708,9,0.444,0.005,0.015
DESC_2_9,-0.552,0.057,-0.664,-0.440,799,0.329,1.072,1.310,1.087,1.552,0.805,0.746,0.769,0.784,0.258,9.303,9,0.752,0.478,0.423
DESC_2_10,1.220,0.110,1.004,1.437,799,0.104,1.509,5.309,1.087,0.461,0.601,0.619,0.771,0.807,0.162,38.075,9,0.000,0.005,0.930


In [9]:
round(pcm.item_restscore_table, 3)

,Observed gamma,Expected gamma,SE,Simulated mean,Simulated SD,z,p,p (BH),p (bootstrap),"p (bootstrap, Westfall-Young)"
DESC_2_1,0.776,0.797,0.017,0.798,0.013,-1.677,0.093,0.187,0.055,0.323
DESC_2_2,0.790,0.798,0.018,0.798,0.014,-0.577,0.564,0.627,0.552,0.806
DESC_2_3,0.801,0.780,0.014,0.781,0.011,1.694,0.090,0.187,0.129,0.438
DESC_2_4,0.773,0.782,0.016,0.784,0.012,-0.911,0.362,0.453,0.423,0.806
DESC_2_5,0.827,0.808,0.015,0.809,0.014,1.301,0.193,0.276,0.114,0.463
DESC_2_6,0.779,0.783,0.016,0.784,0.013,-0.366,0.715,0.715,0.602,0.806
DESC_2_7,0.823,0.795,0.015,0.796,0.012,2.202,0.028,0.138,0.050,0.234
DESC_2_8,0.834,0.782,0.013,0.783,0.012,4.363,0.000,0.000,0.005,0.005
DESC_2_9,0.769,0.784,0.015,0.786,0.012,-1.423,0.155,0.258,0.139,0.463
DESC_2_10,0.771,0.807,0.021,0.809,0.019,-1.973,0.049,0.162,0.045,0.264


The item-restscore *p*-values above test each observed gamma against its simulated distribution. iarm instead uses the normal approximation, *z* = (observed − expected) / SE, with Benjamini–Hochberg adjustment. These are computed from the same table. They agree with iarm's to 3 decimals; at the 4th decimal, iarm's values reflect eRm's looser convergence tolerance.

In [10]:
from scipy.stats import norm

rs = pcm.item_restscore_table
z = (rs['Observed gamma'] - rs['Expected gamma']) / rs['SE']
p = pd.Series(2 * norm.sf(z.abs()), index=rs.index)
ranked = p.sort_values(ascending=False)
p_bh = (ranked * len(p) / np.arange(len(p), 0, -1)).cummin().clip(upper=1).reindex(p.index)

round(pd.DataFrame({'Observed gamma': rs['Observed gamma'],
                    'Expected gamma': rs['Expected gamma'],
                    'SE': rs['SE'],
                    'p (normal)': p,
                    'p (normal, BH)': p_bh}), 4)

,Observed gamma,Expected gamma,SE,p (normal),"p (normal, BH)"
DESC_2_1,0.7759,0.7966,0.0174,0.2345,0.3909
DESC_2_2,0.7902,0.7981,0.0176,0.6561,0.7290
DESC_2_3,0.8006,0.7795,0.0136,0.1212,0.3029
DESC_2_4,0.7729,0.7821,0.0163,0.5720,0.7150
DESC_2_5,0.8268,0.8076,0.0151,0.2027,0.3909
DESC_2_6,0.7790,0.7828,0.0163,0.8121,0.8121
DESC_2_7,0.8232,0.7947,0.0147,0.0532,0.2660
DESC_2_8,0.8345,0.7820,0.0133,0.0001,0.0008
DESC_2_9,0.7694,0.7842,0.0154,0.3372,0.4818
DESC_2_10,0.7715,0.8068,0.0206,0.0859,0.2864


In [11]:
round(pcm.item_q1_table, 3)

,Q1,df,p,p (BH),p (bootstrap),"p (bootstrap, Westfall-Young)"
DESC_2_1,6.108,9,0.729,0.829,0.682,0.945
DESC_2_2,10.168,9,0.337,0.752,0.264,0.881
DESC_2_3,8.802,9,0.456,0.752,0.299,0.881
DESC_2_4,5.941,9,0.746,0.829,0.672,0.945
DESC_2_5,8.077,9,0.526,0.752,0.433,0.900
DESC_2_6,4.323,9,0.889,0.889,0.831,0.945
DESC_2_7,14.055,9,0.120,0.444,0.104,0.517
DESC_2_8,13.708,9,0.133,0.444,0.090,0.448
DESC_2_9,9.303,9,0.410,0.752,0.308,0.881
DESC_2_10,38.075,9,0.000,0.000,0.010,0.015


In [12]:
round(pcm.item_fit_bootstrap, 3)

,Infit MS,Infit 95% low,Infit 95% high,Infit p,Infit p (Westfall-Young),Outfit MS,Outfit 95% low,Outfit 95% high,Outfit p,Outfit p (Westfall-Young)
DESC_2_1,1.097,0.892,1.088,0.070,0.388,1.235,0.838,1.142,0.010,0.090
DESC_2_2,1.101,0.890,1.099,0.065,0.388,1.155,0.776,1.220,0.204,0.687
DESC_2_3,0.892,0.892,1.087,0.045,0.229,0.908,0.897,1.080,0.060,0.353
DESC_2_4,1.065,0.908,1.098,0.254,0.478,1.068,0.912,1.108,0.224,0.687
DESC_2_5,0.898,0.882,1.131,0.104,0.468,0.914,0.809,1.233,0.433,0.801
DESC_2_6,0.979,0.904,1.097,0.662,0.662,1.013,0.902,1.121,0.741,0.930
DESC_2_7,0.910,0.885,1.101,0.164,0.478,0.849,0.865,1.159,0.060,0.353
DESC_2_8,0.799,0.909,1.084,0.005,0.005,0.797,0.908,1.114,0.005,0.015
DESC_2_9,1.072,0.906,1.091,0.149,0.478,1.087,0.910,1.093,0.075,0.423
DESC_2_10,1.509,0.864,1.137,0.005,0.005,1.087,0.660,1.407,0.836,0.930


#### 4. Local dependence and dimensionality

Q3\* (residual correlations minus their mean), with the simulated cutoff, and the first contrast of the residuals' principal components:

In [13]:
pcm.res_corr_analysis()
round(pcm.q3_star, 3)

,DESC_2_1,DESC_2_2,DESC_2_3,DESC_2_4,DESC_2_5,DESC_2_6,DESC_2_7,DESC_2_8,DESC_2_9,DESC_2_10
DESC_2_1,1.111,0.022,-0.073,-0.058,0.191,-0.015,-0.031,-0.134,-0.074,0.095
DESC_2_2,0.022,1.111,-0.054,0.001,-0.084,0.091,-0.063,-0.060,0.096,0.001
DESC_2_3,-0.073,-0.054,1.111,0.049,-0.079,-0.085,0.100,0.226,-0.020,0.020
DESC_2_4,-0.058,0.001,0.049,1.111,-0.102,0.051,0.036,0.051,0.032,-0.123
DESC_2_5,0.191,-0.084,-0.079,-0.102,1.111,0.062,-0.050,0.042,-0.028,0.095
DESC_2_6,-0.015,0.091,-0.085,0.051,0.062,1.111,0.001,-0.014,0.004,-0.086
DESC_2_7,-0.031,-0.063,0.100,0.036,-0.050,0.001,1.111,0.084,-0.079,0.030
DESC_2_8,-0.134,-0.060,0.226,0.051,0.042,-0.014,0.084,1.111,-0.003,-0.063
DESC_2_9,-0.074,0.096,-0.020,0.032,-0.028,0.004,-0.079,-0.003,1.111,-0.005
DESC_2_10,0.095,0.001,0.020,-0.123,0.095,-0.086,0.030,-0.063,-0.005,1.111


In [14]:
round(pcm.q3_bootstrap, 3)

,Item 1,Item 2,Q3,Expected Q3,95% cutoff,p,p (Westfall-Young)
0,DESC_2_3,DESC_2_8,0.115,-0.122,-0.053,0.005,0.005
1,DESC_2_1,DESC_2_5,0.080,-0.124,-0.041,0.005,0.005
2,DESC_2_3,DESC_2_7,-0.011,-0.115,-0.045,0.005,0.244
3,DESC_2_2,DESC_2_9,-0.015,-0.104,-0.037,0.015,0.532
4,DESC_2_1,DESC_2_10,-0.015,-0.106,-0.032,0.025,0.582
5,DESC_2_5,DESC_2_10,-0.016,-0.113,-0.036,0.025,0.652
6,DESC_2_2,DESC_2_6,-0.019,-0.111,-0.045,0.025,0.642
7,DESC_2_7,DESC_2_8,-0.026,-0.115,-0.050,0.015,0.488
8,DESC_2_5,DESC_2_6,-0.048,-0.117,-0.028,0.075,0.915
9,DESC_2_4,DESC_2_6,-0.060,-0.112,-0.045,0.114,0.975


In [15]:
pcm.residual_contrast_bootstrap

{'first_contrast': 1.538995452811947,
 'expected': 1.2901369505390172,
 'cutoff_95': 1.3488468714090416,
 'p': 0.004975124378109453}

Partial gamma tests of local dependence (Kreiner & Christensen, 2004):

In [16]:
round(pcm.local_dependence_gamma(), 3)

,Item,Paired item,Gamma,z,p,p (BH),Flagged
0,DESC_2_3,DESC_2_8,0.491,7.009,0.000,0.000,True
1,DESC_2_8,DESC_2_3,0.445,6.160,0.000,0.000,True
2,DESC_2_1,DESC_2_5,0.308,4.365,0.000,0.000,True
3,DESC_2_5,DESC_2_1,0.295,4.068,0.000,0.001,True
4,DESC_2_4,DESC_2_7,0.217,3.093,0.002,0.020,True
...,...,...,...,...,...,...,...
85,DESC_2_3,DESC_2_1,-0.242,-3.168,0.002,0.017,True
86,DESC_2_10,DESC_2_4,-0.245,-2.687,0.007,0.050,True
87,DESC_2_4,DESC_2_10,-0.283,-3.242,0.001,0.015,True
88,DESC_2_4,DESC_2_5,-0.287,-3.571,0.000,0.005,True


#### 5. Differential item functioning

DIF by gender, with the conditional likelihood ratio omnibus test:

In [17]:
pcm.dif_test('gender',
             seed=1)
round(pcm.dif_table, 3)

/Users/markelliott/Documents/GitHub/RaschPy/raschpy/_progress.py:67: UserWarning: 1 non-extreme person(s) have a missing value for 'gender' and were excluded. Group sizes used: {'male': 348, 'female': 322}.
  return func(*args, **kwargs)


,Group,Reference,Focal,Focal (purified),Difference,SE,t,p,p (corrected),Selected,Flagged,df
Item,,,,,,,,,,,,
DESC_2_1,female,0.114,0.114,0.246,0.132,0.146,0.902,0.367,0.487,True,False,661.786
DESC_2_2,female,0.275,0.591,0.708,0.433,0.165,2.629,0.009,0.080,False,False,668.000
DESC_2_3,female,-0.770,-1.061,-0.972,-0.202,0.125,-1.612,0.107,0.316,True,False,665.375
DESC_2_4,female,-0.548,-0.579,-0.449,0.099,0.125,0.795,0.427,0.487,True,False,665.247
DESC_2_5,female,0.335,0.353,0.490,0.154,0.140,1.099,0.272,0.454,True,False,647.713
DESC_2_6,female,0.124,0.164,0.300,0.175,0.146,1.202,0.230,0.454,True,False,654.566
DESC_2_7,female,0.042,-0.155,-0.052,-0.094,0.121,-0.775,0.438,0.487,True,False,667.139
DESC_2_8,female,-0.045,-0.346,-0.255,-0.210,0.137,-1.531,0.126,0.316,True,False,574.588
DESC_2_9,female,-0.486,-0.596,-0.478,0.008,0.142,0.057,0.955,0.955,True,False,643.404


In [18]:
pcm.dif_omnibus_table

,LR,df,p,Flagged
female,46.728147,9,0.0,True


#### 6. Person fit

Each person's conditional *p*-value: the probability, given their total score, of a response pattern as unlikely as theirs:

In [19]:
p = pcm.person_conditional_p
print(f"{(p < 0.05).sum()} of {p.notna().sum()} persons have p < 0.05")
round(p.sort_values().head(10), 4)

59 of 671 persons have p < 0.05


171    0.001
60     0.001
272    0.001
129    0.001
495    0.001
512    0.001
82     0.001
527    0.001
327    0.002
58     0.002
Name: Conditional p, dtype: float64

#### 7. Targeting and reliability

In [20]:
pcm.targeting()
pcm.targeting_summary

Population mean                  -1.998211
Population SD                     2.192280
Target                            0.013988
Target - population mean          2.012199
Maximum information               8.432572
Expected information              4.228664
Expected / maximum information    0.501468
Test difficulty                   0.032278
Expected |bias|                   0.129347
Expected SEM                      0.529393
Expected RMSE                     0.605935
rho1                              0.959878
rho2                              0.959878
rho3                              0.926787
rho4                              0.926787
rho5                              0.950090
rho6                              0.889553
Name: Targeting, dtype: float64

#### 8. Summary

The results above in one table:

In [21]:
pcm.global_stats_df()

,Section,Value,df,p,Details
Statistic,,,,,
Overall Q1 (item scores),Overall fit,106.704,NaN,0.010,"10 ability groups; Monte Carlo p, 99 samples"
Overall Q1 (thresholds),Overall fit,677.905,NaN,0.020,"10 ability groups; Monte Carlo p, 99 samples"
Andersen LR,Overall fit,38.683,21.0,0.064,split=score; p_method=monte-carlo; unused_cate...
Andersen LR,Overall fit,79.437,39.0,0.000,split=exogenous; covariate=gender
Martin-Lof,Dimensionality and local dependence,351.390,399.0,0.002,split=median; p_method=monte-carlo; 91% of sco...
First residual contrast,Dimensionality and local dependence,1.539,NaN,0.005,bootstrap 95% cutoff 1.349
Q3* (largest),Dimensionality and local dependence,0.226,NaN,NaN,conditional residuals
Item pairs with local dependence,Dimensionality and local dependence,6.000,NaN,NaN,"of 90 ordered pairs; partial gamma, BH p <= 0...."
Items flagged by Q1,Item fit,1.000,NaN,NaN,of 10; BH p <= 0.05


#### 9. Partial credit or rating scale model?

The rating scale model is nested in the partial credit model; with CML estimates, `model_selection(test='LR')` tests it by a conditional likelihood ratio test (the default, `test=None`, uses AIC). With many persons any difference between the two is significant, so the partial credit model is preferred only if a close-fit test also rejects the rating scale model: its null hypothesis is that the partial credit model's thresholds depart from the rating scale model's by at most `delta` (default 0.1 median category widths), tested on the root mean square departure over all the items' thresholds (`p-value (close fit)`) and on the item that departs most (`per item`). `Delta-hat` estimates the departure, in median category widths:

In [22]:
rsm = RSM(responses,
          max_score=4,
          exogenous=exogenous)
rsm.calibrate(estimation='cml')
rsm.model_selection(test='LR')
rsm.model_comparison_rsm_pcm_lr_summary

LR statistic                     286.473231
df                                       27
p-value                                 0.0
delta                                   0.1
p-value (close fit)                     0.0
p-value (close fit, per item)           0.0
Delta-hat                           0.28174
Units                                 width
Preferred                               PCM
Name: RSM vs PCM LR test, dtype: object

#### References

Christensen, K. B., Kreiner, S. & Mesbah, M. (Eds.) (2013). *Rasch Models in Health*. Wiley.

Kreiner, S. & Christensen, K. B. (2004). Analysis of local dependence and multidimensionality in graphical loglinear Rasch models. *Communications in Statistics – Theory and Methods*, 33, 1239–1276.

Mueller, M. (2022). *iarm: Item Analysis in Rasch Models*. R package version 0.4.3. https://CRAN.R-project.org/package=iarm